## Step 1: Import libraries

In [5]:
import ast
import difflib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)

In [4]:
import sys
!{sys.executable} -m pip install nltk

  Using cached nltk-3.10.4-py3-none-any.whl.metadata (5.0 kB)
  Using cached defusedxml-0.7.1-py2.py3-none-any.whl.metadata (32 kB)
Using cached nltk-3.10.4-py3-none-any.whl (1.7 MB)
Using cached defusedxml-0.7.1-py2.py3-none-any.whl (25 kB)

   ---------------------------------------- 0/3 [regex]
   ------------- -------------------------- 1/3 [defusedxml]
   ------------- -------------------------- 1/3 [defusedxml]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   --------------------------

In [8]:
movies = pd.read_csv("dataset/tmdb_5000_movies.csv")
credits = pd.read_csv("dataset/tmdb_5000_credits.csv")

credits = credits.rename(columns={"movie_id": "id"}).drop(columns="title")
df = movies.merge(credits, on="id")

# vote_average / vote_count are used later for quality filtering
df = df[["id", "title", "overview", "genres", "keywords", "cast", "crew",
         "vote_average", "vote_count"]]
print("Shape:", df.shape)
df.head(3)

Shape: (4803, 9)


,id,title,overview,genres,keywords,cast,crew,vote_average,vote_count
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",7.2,11800
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",6.9,4500
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de...",6.3,4466


In [9]:
print("Missing values:\n", df.isnull().sum(), "\n")

df["overview"] = df["overview"].fillna("")
df = df.drop_duplicates(subset="title").reset_index(drop=True)
print("After cleaning:", df.shape)

Missing values:
 id              0
title           0
overview        3
genres          0
keywords        0
cast            0
crew            0
vote_average    0
vote_count      0
dtype: int64 

After cleaning: (4800, 9)


In [10]:
def get_names(text, top_n=None):
    try:
        names = [item["name"] for item in ast.literal_eval(text)]
    except (ValueError, SyntaxError):
        return []
    return names[:top_n] if top_n else names

def get_director(text):
    try:
        return [p["name"] for p in ast.literal_eval(text) if p["job"] == "Director"]
    except (ValueError, SyntaxError):
        return []

def squash(names):
    return [n.replace(" ", "").lower() for n in names]

df["genres"]   = df["genres"].apply(get_names).apply(squash)
df["keywords"] = df["keywords"].apply(get_names).apply(squash)
df["cast"]     = df["cast"].apply(lambda x: get_names(x, top_n=3)).apply(squash)
df["director"] = df["crew"].apply(get_director).apply(squash)

df["metadata"] = df.apply(
    lambda r: " ".join(r["genres"] + r["keywords"] + r["cast"] + r["director"]), axis=1
)
df[["title", "metadata"]].head(3)

,title,metadata
0,Avatar,action adventure fantasy sciencefiction cultur...
1,Pirates of the Caribbean: At World's End,adventure fantasy action ocean drugabuse exoti...
2,Spectre,action adventure crime spy basedonnovel secret...


In [11]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

def preprocess_text(text):
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t.isalpha()]
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    tokens = [stemmer.stem(t) for t in tokens]
    return " ".join(tokens)

sample = df["overview"][0]
print("BEFORE:", sample[:200])
print("AFTER :", preprocess_text(sample)[:200])

BEFORE: In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization.
AFTER : centuri parapleg marin dispatch moon pandora uniqu mission becom torn follow order protect alien civil


In [12]:
df["overview_clean"] = df["overview"].apply(preprocess_text)

# Feature weighting: genres, keywords & director count double, cast counts once
join = lambda col: df[col].apply(" ".join)
df["tags"] = (
    df["overview_clean"] + " "
    + join("genres") + " " + join("genres") + " "
    + join("keywords") + " " + join("keywords") + " "
    + join("director") + " " + join("director") + " "
    + join("cast")
)
df[["title", "tags"]].head(3)

,title,tags
0,Avatar,centuri parapleg marin dispatch moon pandora u...
1,Pirates of the Caribbean: At World's End,captain barbossa long believ dead come back li...
2,Spectre,cryptic messag bond past send trail uncov sini...


In [13]:
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(df["tags"])
print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (4800, 10000)


In [14]:
C = df["vote_average"].mean()
m = df["vote_count"].quantile(0.60)

df["weighted_rating"] = (
    df["vote_count"] / (df["vote_count"] + m) * df["vote_average"]
    + m / (df["vote_count"] + m) * C
)
print(f"Mean rating C = {C:.2f}, vote threshold m = {m:.0f}")

Mean rating C = 6.09, vote threshold m = 370


In [15]:
title_to_idx = pd.Series(df.index, index=df["title"].str.lower())
MIN_VOTES = 100

def find_index(title):
    key = title.lower().strip()
    if key not in title_to_idx.index:
        match = difflib.get_close_matches(key, title_to_idx.index, n=1, cutoff=0.6)
        if not match:
            print(f"'{title}' not found - skipped.")
            return None
        print(f"'{title}' not found. Using '{match[0].title()}' instead.")
        key = match[0]
    return int(title_to_idx[key])

def recommend(titles, n=10, quality=True):
    if isinstance(titles, str):
        titles = [titles]
    idxs = [i for i in (find_index(t) for t in titles) if i is not None]
    if not idxs:
        return "No matching movies found in the dataset."

    # average the vectors of all liked movies, then compare with every movie
    user_vec = np.asarray(tfidf_matrix[idxs].mean(axis=0))
    scores = cosine_similarity(user_vec, tfidf_matrix).flatten()
    scores[idxs] = -1                                  # never recommend the input movies

    top = scores.argsort()[::-1][: 50 if quality else n]
    res = pd.DataFrame({
        "Movie": df["title"].iloc[top].values,
        "Similarity": scores[top],
        "Rating": df["vote_average"].iloc[top].values,
        "Votes": df["vote_count"].iloc[top].values,
        "wr": df["weighted_rating"].iloc[top].values,
    })

    if quality:
        res = res[res["Votes"] >= MIN_VOTES].copy()
        res["final"] = 0.7 * (res["Similarity"] / res["Similarity"].max()) + 0.3 * (res["wr"] / 10)
        res = res.sort_values("final", ascending=False)

    res = res.head(n).drop(columns=["wr", "final"], errors="ignore")
    res["Similarity"] = res["Similarity"].round(3)
    return res.reset_index(drop=True)

In [16]:
recommend("Avatar")

,Movie,Similarity,Rating,Votes
0,Star Trek Into Darkness,0.222,7.4,4418
1,Aliens,0.210,7.7,3220
2,Ender's Game,0.166,6.6,2303
3,Treasure Planet,0.152,7.2,948
4,Titan A.E.,0.158,6.3,313
5,Meet Dave,0.159,5.0,371
6,Small Soldiers,0.147,6.2,511
7,Jimmy Neutron: Boy Genius,0.149,5.6,231
8,Home,0.138,6.8,1519
9,The Fifth Element,0.130,7.3,3885


In [17]:
# same movie, but WITHOUT quality filtering - compare with the output above
recommend("Avatar", quality=False)

,Movie,Similarity,Rating,Votes
0,Star Trek Into Darkness,0.222,7.4,4418
1,Aliens,0.210,7.7,3220
2,Ender's Game,0.166,6.6,2303
3,Meet Dave,0.159,5.0,371
4,Titan A.E.,0.158,6.3,313
5,Treasure Planet,0.152,7.2,948
6,Jimmy Neutron: Boy Genius,0.149,5.6,231
7,Small Soldiers,0.147,6.2,511
8,Galaxina,0.140,3.3,16
9,Sheena,0.140,5.0,22


In [18]:
# multi-movie input: recommendations for someone who likes all three
recommend(["Avatar", "Interstellar", "Inception"], n=8)

,Movie,Similarity,Rating,Votes
0,2001: A Space Odyssey,0.200,7.9,2998
1,Silent Running,0.199,6.3,177
2,Solaris,0.187,7.7,357
3,Star Trek Into Darkness,0.182,7.4,4418
4,Aliens,0.171,7.7,3220
5,Alien,0.163,7.9,4470
6,Armageddon,0.173,6.4,2482
7,Guardians of the Galaxy,0.160,7.9,9742


In [19]:
recommend("The Dark Knight Rises", n=5)

,Movie,Similarity,Rating,Votes
0,The Dark Knight,0.470,8.2,12002
1,Batman Begins,0.380,7.5,7359
2,Batman Returns,0.372,6.6,1673
3,Batman,0.224,7.0,2096
4,Batman Forever,0.241,5.2,1498


In [20]:
recommend("Toy Story", n=5)

,Movie,Similarity,Rating,Votes
0,Toy Story 2,0.462,7.3,3806
1,Toy Story 3,0.412,7.6,4597
2,The 40 Year Old Virgin,0.192,6.2,1983
3,Monster House,0.189,6.3,874
4,"Monsters, Inc.",0.155,7.5,5996
